
# PySpark DataFrame DSL — Part 4
# Column Operations: `lit()`, Rename, Drop, Replace, Cast

## Learning Objectives

By the end of this part, you should be able to:

- Add constant values using `lit()`.
- Rename columns using `withColumnRenamed()` and `alias()`.
- Remove columns using `drop()`.
- Replace matching values using `replace()`.
- Convert column types using `cast()`.
- Explain each operation's purpose and parameters.
- Combine the operations into an ETL transformation pipeline.

> Run the setup section first. Each numbered example is intended to be run independently unless noted otherwise.

---

# 0. Setup: Example DataFrame

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when

spark = SparkSession.builder \
    .appName("DataFrameDSLPart4") \
    .getOrCreate()

data = [
    ("C001", "Karthik", "Chennai", "25", "55000.50", "Active"),
    ("C002", "Arun", "Mumbai", "28", "62000.75", "Active"),
    ("C003", "Rahul", "Delhi", "26", "48000.25", "Inactive"),
    ("C004", "Priya", "Bangalore", "30", "75000.00", "Active"),
    ("C005", "Anitha", "Chennai", "24", "45000.00", "Pending"),
    ("C006", "Vijay", "Mumbai", "32", "82000.00", "Inactive")
]

columns = ["customer_id", "name", "city", "age", "salary", "status"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)
df.printSchema()
```

**Notice:** `age` and `salary` are strings in this dataset. We'll use them to demonstrate casting.

---

# 1. `lit()` — Create a Literal / Constant Column

## Definition

`lit()` creates a literal value represented as a Spark Column expression. It is commonly used with `withColumn()` to add the same constant value to every row.

```python
from pyspark.sql.functions import lit
```

### Syntax and parameter

```python
lit(value)
```

- `value`: the constant value to represent, such as a string, integer, decimal number, Boolean, or `None` (NULL).
- Return value: a Spark Column expression.
- `lit()` alone does not add a column to a DataFrame. Put it inside `withColumn()` or `select()`.

## Example 1 — Add a country column

```python
df1 = df.withColumn("country", lit("India"))
df1.show()
```

**Parameters explained:**
- `"country"` is the output column name supplied to `withColumn()`.
- `lit("India")` creates a constant string expression.
- Every row receives `India`; the original `df` is unchanged.

## Example 2 — Add a numeric constant

```python
df1 = df.withColumn("bonus", lit(5000))
df1.show()
```

- `"bonus"` is the new column name.
- `lit(5000)` creates an integer literal.
- Every row receives `5000`. This example demonstrates a fixed bonus, not a business rule for calculating actual bonuses.

## Example 3 — Add a decimal constant

```python
df1 = df.withColumn("tax_rate", lit(0.18))
df1.show()
```

- `"tax_rate"` names the new column.
- `lit(0.18)` creates a numeric literal.
- `0.18` represents 18% when treated as a rate; `lit()` does not format it as the text `18%`.

## Example 4 — Add a Boolean flag

```python
df1 = df.withColumn("is_processed", lit(False))
df1.show()
```

- `"is_processed"` is the new column.
- `lit(False)` creates a Boolean value.
- Every record starts with `False`.

## Example 5 — Add a source-system label

```python
df1 = df.withColumn("source_system", lit("CRM"))
df1.show()
```

- `"source_system"` names the metadata column.
- `"CRM"` is the source label.
- This pattern is useful for tagging records with their origin during ETL.

## Example 6 — Add a batch identifier

```python
df1 = df.withColumn("batch_id", lit("BATCH_2026_10_11"))
df1.show()
```

- `"batch_id"` is the column name.
- The string passed to `lit()` is the batch label.
- All rows get the same value in this example.

## Example 7 — Use a literal in `select()`

```python
df.select(
    "customer_id",
    "name",
    lit("India").alias("country")
).show()
```

- `select()` defines the output columns.
- `lit("India")` creates the constant expression.
- `.alias("country")` assigns the expression's output name.
- This displays a literal column in the result without assigning a new DataFrame to `df`.

## Example 8 — Add a constant, then calculate with it

```python
df1 = df.withColumn("bonus", lit(5000)) \
        .withColumn(
            "salary_with_bonus",
            col("salary").cast("double") + col("bonus")
        )
df1.show()
```

- The first `withColumn()` adds `bonus`.
- `col("salary").cast("double")` converts the string salary into a numeric expression.
- `col("bonus")` references the new column.
- The second `withColumn()` adds the two numeric values.

## Example 9 — Use a Python variable as a literal

```python
environment = "development"

df1 = df.withColumn("environment", lit(environment))
df1.show()
```

- `environment` is a normal Python variable.
- `lit(environment)` wraps its current value in a Spark Column expression.
- The same value is used for every row; it is not recalculated per row.

## Example 10 — Create a typed NULL literal

```python
df1 = df.withColumn(
    "optional_note",
    lit(None).cast("string")
)
df1.show()
df1.printSchema()
```

- `lit(None)` represents NULL.
- `.cast("string")` gives the NULL expression an explicit string type.
- Every row has NULL in `optional_note`.
- `None`, `"None"`, and `"NULL"` are different: only `None` here represents a NULL value.

### `lit()` key points

- `lit(value)` creates a Column expression, not a DataFrame.
- `withColumn("x", lit(value))` adds a constant column.
- `select(lit(value).alias("x"))` includes a literal in a selected result.
- Use compatible types when doing calculations.

---

# 2. Rename — Change Column Names

## Definition

Renaming changes a column's name while retaining its data.

Common options:

- `withColumnRenamed(existing, new)`: rename an existing DataFrame column.
- `alias(name)`: give a selected column or expression a name in the result.

## `withColumnRenamed()` syntax and parameters

```python
df.withColumnRenamed(existing, new)
```

- `existing`: current column name.
- `new`: desired column name.
- Returns a new DataFrame.
- If the old name is not found, Spark generally leaves the DataFrame unchanged, so check the schema when correctness matters.

## Example 1 — Rename `name`

```python
df1 = df.withColumnRenamed("name", "customer_name")
df1.show()
```

- `"name"` is the existing name.
- `"customer_name"` is the new name.
- The values remain unchanged; only the column label changes.

## Example 2 — Rename `city`

```python
df1 = df.withColumnRenamed("city", "customer_city")
df1.show()
```

- `"city"` identifies the current column.
- `"customer_city"` is the new name.
- City values such as Chennai and Mumbai remain unchanged.

## Example 3 — Rename multiple columns

```python
df1 = df \
    .withColumnRenamed("name", "customer_name") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("status", "customer_status")

df1.show()
```

- Each call renames one column.
- Each call returns a DataFrame, so the next rename can be chained.
- The final DataFrame contains all three new names.

## Example 4 — Rename an identifier

```python
df1 = df.withColumnRenamed("customer_id", "id")
df1.printSchema()
```

- `"customer_id"` is the old identifier.
- `"id"` is the new, shorter name.
- `printSchema()` verifies the resulting column name.

## Example 5 — Use a business-friendly name

```python
df1 = df.withColumnRenamed("salary", "monthly_salary")
df1.show()
```

- `"salary"` is the source column name.
- `"monthly_salary"` clarifies the intended meaning.
- Renaming does not convert the column's data type.

## Example 6 — Rename, then select

```python
df1 = df \
    .withColumnRenamed("name", "customer_name") \
    .select("customer_id", "customer_name", "city")

df1.show()
```

- The rename happens first.
- `select()` then retains only the three named columns.
- It uses `"customer_name"` because that is now the column name.

## Example 7 — Use `alias()` in `select()`

```python
df.select(
    col("customer_id").alias("id"),
    col("name").alias("customer_name"),
    col("city")
).show()
```

- `col("customer_id")` references the existing column.
- `.alias("id")` names it in the output.
- The same applies to `name`.
- The original `df` is not renamed.

## Example 8 — Alias a calculated expression

```python
df.select(
    col("name"),
    (col("age").cast("int") + 1).alias("next_year_age")
).show()
```

- `col("age")` references the age column.
- `.cast("int")` converts the string age into an integer expression.
- `+ 1` calculates the next-year age.
- `.alias("next_year_age")` names the calculated output column.

## Example 9 — Verify names after renaming

```python
df1 = df.withColumnRenamed("status", "customer_status")
print(df1.columns)
```

- `withColumnRenamed()` returns a new DataFrame.
- `.columns` returns its column names as a Python list.
- This is a quick way to confirm the rename.

## Example 10 — Rename, then use the new name

```python
df1 = df \
    .withColumnRenamed("salary", "monthly_salary") \
    .withColumn(
        "annual_salary",
        col("monthly_salary").cast("double") * 12
    )

df1.show()
```

- The first operation renames `salary`.
- The next expression references `monthly_salary`.
- Casting ensures that the string salary is numeric before multiplication.

### Rename key points

- Renaming changes the column name, not its values.
- Use `withColumnRenamed()` to rename an existing DataFrame column.
- Use `alias()` to name a selected column or expression.
- Use the new name in later operations after renaming.
- Verify names with `df.columns` or `df.printSchema()`.

---

# 3. `drop()` — Remove Columns

## Definition

`drop()` returns a DataFrame with specified columns removed.

### Syntax and parameters

```python
df.drop("column_name")
df.drop("column1", "column2")
```

- Pass one or more column names to remove.
- A new DataFrame is returned.
- The original DataFrame is not modified in place.

## Example 1 — Drop one column

```python
df1 = df.drop("status")
df1.show()
```

- `"status"` is the column to remove.
- All other columns remain.
- The original `df` still has `status`.

## Example 2 — Drop multiple columns

```python
df1 = df.drop("city", "status")
df1.show()
```

- Both `"city"` and `"status"` are removed.
- The remaining columns keep their order.

## Example 3 — Drop an identifier

```python
df1 = df.drop("customer_id")
df1.show()
```

- The customer ID is removed from the result.
- Do this only when downstream processing does not need the identifier.

## Example 4 — Add a column, then drop another

```python
df1 = df.withColumn("country", lit("India")) \
        .drop("status")
df1.show()
```

- `withColumn()` adds `country`.
- `drop()` removes `status`.
- The returned DataFrame has `country` but not `status`.

## Example 5 — Rename, then drop

```python
df1 = df \
    .withColumnRenamed("name", "customer_name") \
    .drop("city")

df1.show()
```

- The name column is renamed first.
- The city column is then removed.
- The result retains `customer_name`.

## Example 6 — Remove several unneeded fields

```python
df1 = df.drop("city", "age", "status")
df1.show()
```

- Each listed column is removed.
- This can reduce the columns passed to later processing.
- If it is easier to list what you want to keep, `select()` may be clearer.

## Example 7 — Drop a column name that is absent

```python
df1 = df.drop("unknown_column")
df1.show()
```

- `"unknown_column"` is not present.
- Spark generally leaves the DataFrame unchanged for a missing name.
- Check `df1.columns` to catch misspellings.

## Example 8 — Drop using a variable

```python
column_to_remove = "status"

df1 = df.drop(column_to_remove)
df1.show()
```

- `column_to_remove` is a Python string variable.
- `drop()` receives its value.
- This can be useful for reusable transformations.

## Example 9 — Drop a calculated column

```python
df1 = df \
    .withColumn("annual_salary", col("salary").cast("double") * 12) \
    .drop("annual_salary")

df1.show()
```

- The first operation creates `annual_salary`.
- The second removes it.
- This illustrates the mechanics; usually avoid creating a column that you immediately discard.

## Example 10 — Remove an identifier before a chosen output

```python
df1 = df.drop("customer_id")
df1.select("name", "city", "status").show()
```

- `drop()` removes the ID from the returned DataFrame.
- `select()` then keeps only the listed columns.
- In real projects, follow applicable data governance requirements.

### `drop()` key points

- `drop()` returns a new DataFrame.
- The original remains unchanged.
- One or multiple columns can be dropped.
- A missing column name generally leaves the DataFrame unchanged.
- Use `select()` when it is clearer to list columns to retain.

---

# 4. Replace — Replace Matching Values

## Definition

Replacing values substitutes one value with another. Two common approaches are:

1. `DataFrame.replace()` or `df.na.replace()` for matching values.
2. `when()` for conditional replacements based on a column expression.

For filling actual NULL values, `fillna()` / `na.fill()` is often the appropriate operation.

> **Important distinction:** `replace()` changes matching values; `withColumnRenamed()` changes a column name.

### Common syntax and parameters

```python
df.replace(to_replace, value, subset=None)
```

- `to_replace`: value or dictionary of values to find.
- `value`: replacement value when a single old value is supplied.
- `subset`: optional list of column names where replacement should be attempted.

For multiple replacements, a dictionary is convenient. Replacement values must be compatible with the column data types.

## Example 1 — Replace one status value

```python
df1 = df.replace("Inactive", "Closed", subset=["status"])
df1.show()
```

- `"Inactive"` is the value to find.
- `"Closed"` is its replacement.
- `subset=["status"]` restricts replacement to the status column.
- Other status values remain unchanged.

## Example 2 — Replace a different status label

```python
df1 = df.replace("Pending", "In Progress", subset=["status"])
df1.show()
```

- `"Pending"` is the target value.
- `"In Progress"` is the new value.
- The subset prevents the operation from targeting unrelated columns.

## Example 3 — Replace several values using a mapping

```python
replacement_map = {
    "Active": "A",
    "Inactive": "I",
    "Pending": "P"
}

df1 = df.replace(replacement_map, subset=["status"])
df1.show()
```

- The dictionary maps old values to new values.
- `subset=["status"]` limits replacements to status.
- This is useful for standardizing category codes.

## Example 4 — Replace city labels

```python
df1 = df.replace(
    {"Chennai": "MAA", "Mumbai": "BOM"},
    subset=["city"]
)
df1.show()
```

- The dictionary specifies two substitutions.
- Only the city column is targeted.
- Use abbreviations only when they are the intended representation.

## Example 5 — Replace an integer value

```python
age_df = df.withColumn("age_int", col("age").cast("int"))

age_df1 = age_df.replace(25, 26, subset=["age_int"])
age_df1.show()
```

- The cast creates an integer column from string age values.
- `25` is the value to find.
- `26` is the replacement.
- `subset` restricts the operation to `age_int`.

## Example 6 — Replace multiple numeric values

```python
age_df1 = age_df.replace(
    {24: 25, 26: 27},
    subset=["age_int"]
)
age_df1.show()
```

- `24` maps to `25`.
- `26` maps to `27`.
- Only `age_int` is targeted.
- These are illustrative replacements; only use such corrections when the business rule justifies them.

## Example 7 — Replace conditionally with `when()`

```python
df1 = df.withColumn(
    "standard_status",
    when(col("status") == "Inactive", "Closed")
    .otherwise(col("status"))
)
df1.show()
```

- `when()` tests each row's status.
- If it is `Inactive`, the new column contains `Closed`.
- `otherwise(col("status"))` preserves every other original status.
- This creates a new column rather than replacing the existing status column.

## Example 8 — Map multiple categories using conditions

```python
df1 = df.withColumn(
    "status_code",
    when(col("status") == "Active", "A")
    .when(col("status") == "Inactive", "I")
    .when(col("status") == "Pending", "P")
    .otherwise("U")
)
df1.show()
```

- The first condition maps Active to A.
- The second maps Inactive to I.
- The third maps Pending to P.
- `otherwise("U")` handles any value not covered by those conditions.

## Example 9 — Convert a placeholder string to NULL

```python
df1 = df.withColumn(
    "city_clean",
    when(col("city") == "UNKNOWN", lit(None).cast("string"))
    .otherwise(col("city"))
)
df1.show()
```

- The condition checks for the literal text `"UNKNOWN"`.
- Matching rows receive a typed string NULL.
- Other rows retain the original city.
- This handles a placeholder string, not all existing NULL values.

## Example 10 — Replace, then rename the column

```python
df1 = df \
    .replace("Inactive", "Closed", subset=["status"]) \
    .withColumnRenamed("status", "customer_status")

df1.show()
```

- `replace()` changes matching values.
- `withColumnRenamed()` changes the column name.
- The operations affect different things: values versus schema names.

### `replace()` key points

- `replace()` substitutes matching values.
- `subset` limits where replacement occurs.
- A dictionary can define multiple substitutions.
- Use `when()` for more complex conditional rules.
- Use `fillna()` / `na.fill()` to fill actual NULL values.
- Check data types when replacing numeric values.

---

# 5. `cast()` — Convert a Column's Data Type

## Definition

`cast()` converts a column expression to a target Spark SQL data type. It is useful when a source column has the wrong type for the operation you want to perform.

### Syntax and parameters

```python
col("column_name").cast("target_type")
```

- `col("column_name")`: the column expression to convert.
- `"target_type"`: the desired type, such as `"int"`, `"double"`, `"string"`, `"boolean"`, or `"date"`. You may also use a Spark type object such as `IntegerType()`.
- `cast()` returns a Column expression. Use `withColumn()` to replace a column or `select()` to return the converted expression.

## Example 1 — String to integer

```python
df1 = df.withColumn(
    "age",
    col("age").cast("int")
)

df1.printSchema()
df1.show()
```

- `col("age")` references the current age column.
- `.cast("int")` converts values like `"25"` to integer `25`.
- `withColumn("age", ...)` replaces age in the returned DataFrame.
- The original `df` is unchanged.

## Example 2 — String to double

```python
df1 = df.withColumn(
    "salary",
    col("salary").cast("double")
)

df1.printSchema()
df1.show()
```

- `col("salary")` references the string salary.
- `.cast("double")` converts values such as `"55000.50"` to a numeric double.
- This enables calculations such as salary multiplied by 12.

## Example 3 — Cast while retaining the original

```python
df1 = df.select(
    "customer_id",
    "salary",
    col("salary").cast("double").alias("salary_numeric")
)

df1.show()
```

- The original string `salary` is selected unchanged.
- The expression is separately cast to double.
- `alias("salary_numeric")` names the converted result.

## Example 4 — Cast an integer expression to string

```python
df1 = df.withColumn(
    "age_text",
    col("age").cast("int").cast("string")
)

df1.show()
df1.printSchema()
```

- The first cast converts age to integer.
- The second cast converts that expression to string.
- This demonstrates chained casts; it is not necessary if the goal is simply to keep the original string age.

## Example 5 — Cast before arithmetic

```python
df1 = df.withColumn(
    "annual_salary",
    col("salary").cast("double") * 12
)

df1.show()
```

- Salary is initially a string.
- `.cast("double")` converts it to a numeric expression.
- Multiplying by 12 calculates annual salary.

## Example 6 — Cast using a type object

```python
from pyspark.sql.types import IntegerType

df1 = df.withColumn(
    "age_int",
    col("age").cast(IntegerType())
)

df1.printSchema()
```

- `IntegerType()` is a Spark SQL type object.
- `cast(IntegerType())` uses that type as the target.
- This is an alternative to `.cast("int")`.

## Example 7 — Cast to Boolean

```python
bool_df = spark.createDataFrame(
    [("true",), ("false",)],
    ["flag_text"]
)

bool_df1 = bool_df.withColumn(
    "flag",
    col("flag_text").cast("boolean")
)

bool_df1.show()
bool_df1.printSchema()
```

- `flag_text` contains string representations of Boolean values.
- `.cast("boolean")` attempts to convert them to Boolean values.
- Validate actual source strings in production; unexpected values may not convert as intended.

## Example 8 — Cast to date

```python
date_df = spark.createDataFrame(
    [("2026-10-11",), ("2026-01-05",)],
    ["date_text"]
)

date_df1 = date_df.withColumn(
    "order_date",
    col("date_text").cast("date")
)

date_df1.show()
date_df1.printSchema()
```

- `date_text` is a string.
- `.cast("date")` converts compatible date strings to Spark date values.
- A standard representation such as `yyyy-MM-dd` is suitable for this example.

## Example 9 — Invalid numeric string

```python
invalid_df = spark.createDataFrame(
    [("100",), ("not_a_number",)],
    ["amount_text"]
)

invalid_df1 = invalid_df.withColumn(
    "amount",
    col("amount_text").cast("int")
)

invalid_df1.show()
```

- `"100"` is a valid integer representation.
- `"not_a_number"` is not.
- Depending on Spark version and ANSI mode, an invalid cast may produce NULL or raise an exception.
- Production pipelines should validate or explicitly handle malformed input.

## Example 10 — Cast multiple columns in a pipeline

```python
df1 = df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("salary", col("salary").cast("double")) \
    .withColumn("annual_salary", col("salary") * 12)

df1.printSchema()
df1.show()
```

- The first operation converts age to integer.
- The second converts salary to double.
- The third uses the numeric salary to calculate annual salary.
- Chaining makes the sequence of schema transformations explicit.

### `cast()` key points

- `cast()` changes data type, not column name.
- Use compatible target types such as `"int"`, `"double"`, `"string"`, `"boolean"`, or `"date"`.
- Invalid values may produce NULL or an error, depending on data and Spark settings.
- Verify types using `printSchema()`.
- For complex date/time formats, parsing functions such as `to_date()` may be more appropriate than a simple cast.

---

# 6. Compare the Operations

| Operation | What it changes | Example |
|---|---|---|
| `lit()` | Creates a constant expression | `withColumn("country", lit("India"))` |
| `withColumnRenamed()` | Column name | `withColumnRenamed("name", "customer_name")` |
| `alias()` | Output name of an expression | `col("name").alias("customer_name")` |
| `drop()` | Column presence | `drop("status")` |
| `replace()` | Matching values | `replace("Inactive", "Closed", subset=["status"])` |
| `when()` | Value based on a condition | `when(col("age") >= 18, "Adult")` |
| `cast()` | Data type | `col("age").cast("int")` |

---

# 7. Combined Data Engineering Example

## Requirement

1. Convert `age` to integer.
2. Convert `salary` to double.
3. Rename `name` to `customer_name`.
4. Replace `Inactive` with `Closed`.
5. Add a constant `country` column.
6. Calculate annual salary.
7. Remove `city` from the final output.

```python
from pyspark.sql.functions import col, lit

result = df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("salary", col("salary").cast("double")) \
    .withColumnRenamed("name", "customer_name") \
    .replace("Inactive", "Closed", subset=["status"]) \
    .withColumn("country", lit("India")) \
    .withColumn("annual_salary", col("salary") * 12) \
    .drop("city")

result.printSchema()
result.show(truncate=False)
```

### Step-by-step

```text
Original DataFrame
       |
       v
Cast age to integer
       |
       v
Cast salary to double
       |
       v
Rename name -> customer_name
       |
       v
Replace Inactive -> Closed in status
       |
       v
Add country = India
       |
       v
Calculate annual_salary
       |
       v
Drop city
       |
       v
Final DataFrame
```

### Why order matters

- Salary is cast before arithmetic.
- The renamed column should be referenced by its new name in later expressions.
- `replace()` targets the current `status` column.
- `drop("city")` removes city only from the returned DataFrame.

---

# 8. Common Mistakes

## Mistake 1 — Treating `lit()` as a DataFrame method

Incorrect:

```python
lit("India").show()
```

`lit()` returns a Column expression, not a DataFrame.

Correct:

```python
df.withColumn("country", lit("India")).show()
```

## Mistake 2 — Confusing rename with replacement

```python
df.withColumnRenamed("status", "customer_status")
```

changes a column name.

```python
df.replace("Inactive", "Closed", subset=["status"])
```

changes matching values.

## Mistake 3 — Doing arithmetic on strings

Cast a numeric string first:

```python
col("salary").cast("double") * 12
```

## Mistake 4 — Expecting transformations to mutate `df`

```python
df1 = df.drop("status")
```

The original `df` still contains `status`.

## Mistake 5 — Assuming every cast succeeds

Malformed values may yield NULL or raise an exception depending on Spark settings. Validate source data and inspect the result.

---

# 9. Quick Reference

```python
# Add a constant column
df.withColumn("country", lit("India"))

# Rename a column
df.withColumnRenamed("name", "customer_name")

# Alias a selected expression
df.select(col("name").alias("customer_name"))

# Drop columns
df.drop("status")
df.drop("city", "status")

# Replace matching values
df.replace("Inactive", "Closed", subset=["status"])

# Replace several categorical values
df.replace(
    {"Active": "A", "Inactive": "I", "Pending": "P"},
    subset=["status"]
)

# Cast a column
df.withColumn("age", col("age").cast("int"))

# Add a constant and calculate a value
df.withColumn("country", lit("India")) \
  .withColumn("annual_salary", col("salary").cast("double") * 12)
```

---

# 10. Key Takeaways

1. `lit(value)` creates a literal Column expression; use it with `withColumn()` to add a constant column.
2. `withColumnRenamed(existing, new)` renames an existing column.
3. `alias(name)` names a selected column or expression.
4. `drop()` removes one or more columns from the returned DataFrame.
5. `replace()` substitutes matching values and can be restricted with `subset`.
6. `when()` is useful for conditional replacement rules.
7. `cast()` converts a column expression to a target Spark SQL type.
8. These transformations return new DataFrames; assign the result to use it later.
9. Check schema changes with `printSchema()` and inspect data with `show()`.
10. Use the right operation for the requirement: value, name, column presence, or data type.

## Core Pattern

```python
from pyspark.sql.functions import col, lit, when

result = df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("salary", col("salary").cast("double")) \
    .withColumnRenamed("name", "customer_name") \
    .replace("Inactive", "Closed", subset=["status"]) \
    .withColumn("country", lit("India")) \
    .withColumn(
        "customer_type",
        when(col("salary") >= 70000, "Premium")
        .otherwise("Standard")
    ) \
    .drop("city")

result.show(truncate=False)
```

This combines type conversion, renaming, value replacement, constant columns, conditional expressions, and dropping columns in one transformation pipeline.
